# Week 1 - Day 2: Your First Agent

Today we go from a plain LLM call to a working agent, one small step at a time.

| Part | What we do |
|---|---|
| **1. Basic LLM** | Call a model, give it a short-term memory, and see what it cannot do |
| **2. One tool** | Write a web search tool and use it by hand |
| **3. Workflow** | Fixed steps that we write: search words, search, answer |
| **4. Function calling** | See the model ask for the tool, and how the answer comes back |
| **5. Agent** | Let the model decide the steps, with a proper system prompt and a short-term memory |
| **6. Your turn** | You write your own tool and give it to the agent |

Everything is free: **Groq** for the model and **DuckDuckGo** for the web search. You need one free key (Groq). The search needs no key.

> **Student notebook (live coding).** From Part 1 on, every code cell is a **TODO**. Watch the instructor's screen and **type the code yourself**. The comments tell you what to write and give hints. The setup cells are already complete: just run them.
>
> Tips: run a cell with **Shift + Enter**. A cell may fail until the earlier TODO cells are filled in. That is normal: finish the earlier cell first.

## Setup
1. Get a free **Groq** key at **console.groq.com** -> API Keys.
2. Give the notebook your key:
   - **On Google Colab:** click the Secrets icon (key) in the left bar, add a secret named `GROQ_API_KEY`, paste your key as the value, and switch **Notebook access** on.
   - **On your own computer:** put `GROQ_API_KEY=your_key_here` in a file called `.env` in this folder (or any folder above it).

The web search (DuckDuckGo) needs no key. Your key is like a password: never share it and never type it into the notebook.

**Optional:** if you have a free **LangSmith** key (smith.langchain.com), add it the same way with the name `LANGSMITH_API_KEY`. Every run is then recorded online, so you can open it and see each model call, the tool that was called, its inputs and result, the time and the tokens. Without the key, nothing changes.

In [17]:
%pip install -q langchain langchain-groq ddgs python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
import os
import time
from datetime import date
from dotenv import find_dotenv, load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import InMemorySaver
from ddgs import DDGS

load_dotenv(find_dotenv(usecwd=True), override=True)    # on your computer: reads your .env file
GROQ_API_KEY = os.getenv("GROQ_API_KEY", "")

try:                                                     # on Google Colab: reads the key from Secrets
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY") or GROQ_API_KEY
except Exception:
    pass

if GROQ_API_KEY == "":
    raise RuntimeError("No GROQ_API_KEY found. Add it (Colab: Secrets, computer: .env file), then run this cell again.")

# Optional: record every run in LangSmith (smith.langchain.com). Skipped when there is no key.
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY", "")
try:
    from google.colab import userdata
    LANGSMITH_API_KEY = userdata.get("LANGSMITH_API_KEY") or LANGSMITH_API_KEY
except Exception:
    pass

if LANGSMITH_API_KEY != "":
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
    os.environ["LANGSMITH_PROJECT"] = "day2-first-agent"
    print("LangSmith tracing is ON. Open smith.langchain.com and look for the project day2-first-agent.")

MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-120b")
THINKING = "low" if MODEL.startswith("openai/gpt-oss") else None    # "low" makes gpt-oss models answer faster

llm = ChatGroq(model=MODEL, api_key=GROQ_API_KEY, temperature=0.3, reasoning_effort=THINKING)
today = str(date.today())          # a model does not know today's date, so we will tell it
print("Ready. Model:", MODEL)

LangSmith tracing is ON. Open smith.langchain.com and look for the project day2-first-agent.
Ready. Model: openai/gpt-oss-120b


---
# Part 1 - Basic LLM

One question in, one answer out. `llm.invoke(...)` sends the question to the model and returns a **message object**. The text is in `.content`.

In [19]:
# TODO 1.1: ask the model to say hello to a class of AI students in one short sentence.
#   Hint: reply = llm.invoke("...your question...")

# TODO 1.2: print the text of the reply.
#   Hint: the text is in reply.content

reply = llm.invoke("Say Hello to a class of AI students in one short sentence")
print(reply.content)


Hello, future AI innovators—welcome to the exciting world of intelligent machines!


### What is inside the reply?
The reply is more than text. It also says how many **tokens** were used.

In [20]:
# TODO 1.3: print three things about the reply: its type, its text and its token usage.
#   Hints: type(reply).__name__      reply.content      reply.usage_metadata
#   Print them like this:  print("Type  :", ...)   print("Text  :", ...)   print("Tokens:", ...)

print(reply)
print("Type  :", type(reply).__name__)
print("Text  :", reply.content)
print("Tokens:", reply.usage_metadata)

content='Hello, future AI innovators—welcome to the exciting world of intelligent machines!' additional_kwargs={'reasoning_content': 'We need to comply. Simple.'} response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 83, 'total_tokens': 115, 'completion_time': 0.067953848, 'completion_tokens_details': {'reasoning_tokens': 8}, 'prompt_time': 0.003582872, 'prompt_tokens_details': None, 'queue_time': 0.346557524, 'total_time': 0.07153672}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_0708ac49a5', 'service_tier': 'on_demand', 'reasoning_effort': 'low', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a1173f-2fc3-7ec2-8495-4ea76e887e1f-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 83, 'output_tokens': 32, 'total_tokens': 115, 'output_token_details': {'reasoning': 8}}
Type  : AIMessage
Text  : Hello, future AI innovators—welcome to the exciting world of intelligent machines!
Tokens: {'input_to

### Messages and roles
A chat has three kinds of message:

| Role | Class | Who writes it |
|---|---|---|
| system | `SystemMessage` | You: the rules for the model |
| user | `HumanMessage` | The person asking |
| assistant | `AIMessage` | The model's earlier answers |

We send a **list** of messages. The system message changes how the model behaves.

In [21]:
# TODO 1.4: send a LIST of messages: a SystemMessage that says "You are a pirate. Answer in one short sentence."
#           and a HumanMessage that asks "What is an AI agent?". Then print the text of the reply.
#   Hint: reply = llm.invoke([SystemMessage("..."), HumanMessage("...")])
SystemMessage = "You are a pirate. Answer in one short sentence"
HumanMessage = "What is an AI agent?"
reply = llm.invoke([
    SystemMessage,
    HumanMessage
])

print(reply.content)

An AI agent is a software entity that perceives its environment, makes decisions, and takes actions autonomously to achieve specific goals.


### Short-term memory: a model forgets
A model remembers **nothing** between calls. Every `llm.invoke(...)` starts fresh. Watch: we tell it our name, then ask for it in a **second, separate call**.

In [22]:
# TODO 1.5: show that a model forgets. Make two SEPARATE calls:
#   1. llm.invoke("My name is Aarav. Say hi in five words.")  and print the reply text
#   2. llm.invoke("What is my name? Answer in one short sentence.")  and print the reply text
#   Can the second call know the name?
reply_1 = llm.invoke("My name is Anup. Say hi in five words")
print(reply_1.content)

reply_2 = llm.invoke("What is my name? Answer in one short sentence")
print(reply_2.content)


Hi there, Anup! Nice meeting!
I don’t know your name.


The second call is brand new, so the model cannot know. A chat app seems to remember because it sends the **whole conversation again** every time. We can do the same: keep a **list of messages**, add every question and every answer to it, and send the whole list.

In [23]:
# TODO 1.6: give the model a short-term memory with a LIST of messages.
#   - chat = []
#   - chat.append(HumanMessage("My name is Aarav. Say hi in five words."))     then  reply = llm.invoke(chat)
#   - chat.append(reply)       (remember the answer too)      then print(reply.content)
#   - chat.append(HumanMessage("What is my name? Answer in one short sentence."))     then  reply = llm.invoke(chat)
#   - chat.append(reply)      then print(reply.content)
#   - print("Messages kept:", len(chat))

chat = []
HumanMessage = "My name is Anup Rai. Say hi in five words"
chat.append(HumanMessage)
reply = llm.invoke(chat)
chat.append(reply)
print(reply.content)

HumanMessage = "What is my name? Answer in one short sentence"
chat.append(HumanMessage)
reply = llm.invoke(chat)
chat.append(reply)
print(reply.content)

print("Message kept:", len(chat))

Hi Anup, great to meet!
Your name is Anup Rai.
Message kept: 4


The model did not remember anything: **we** sent the earlier messages again. That list is the conversation's **state**, and keeping it is **short-term memory**. It lasts only while the list exists, and a longer list means more tokens to send each time.

### What a plain LLM cannot do
A model only knows what it learned during training. It has **no internet**, so it cannot look anything up, and it cannot know what happened today. Read the answer: did it search, or did it guess?

In [24]:
# TODO 1.7: ask the model: "Search the web and tell me today's top news headline from Nepal."
#           Print the reply. Did it search, or did it guess?

message = llm.invoke("What is today's the weather of kathmandu?")
print(message.content)

I’m sorry, but I don’t have the ability to retrieve real‑time weather information. You can check the current conditions in Kathmandu on a weather website or app such as Weather.com, AccuWeather, or a local news source.


It cannot search. It either refuses or invents a headline from memory. To look things up, the model needs a **tool**. Let us write one.

---
# Part 2 - One tool

A **tool** is a normal Python function with `@tool` on top. The model never sees the code. It only sees three things: the **name**, the **docstring** (the sentence in triple quotes) and the **inputs**.

Our tool searches the web with DuckDuckGo. It returns the top three results: a title, a short summary and a link.

In [25]:
# TODO 2.1: write the tool.
#   - put @tool on the line above the function
#   - def search_web(query: str) -> str:
#   - the docstring (one sentence) tells the model what the tool does
#   - get the results with:  results = DDGS().text(query, max_results=3)
#     The free search sometimes says "slow down", so try twice: use  for attempt in range(2):  with try / except,
#     break when it works, and time.sleep(2) when it fails.
#   - if results is still None, return "Error: the search failed. ..."
#   - if there are no results, return "Error: nothing found for " + query + ". Try different words."
#   - otherwise return one line per result:  title + ": " + body[:300] + " (" + href + ")"   joined with "\n"
#     (each result is a dictionary with the keys "title", "body" and "href")

@tool
def search_web(query: str) -> str:
    """Search the web and return the top results with a short summary and a link for each."""
    if not query.strip():
        return "Error: search query cannot be empty."

    for attempt in range(2):
        try:
            results = DDGS().text(query, max_results=3)

            if not results:
                return "Error: nothing found for " + query + ". Try different words."

            lines = []
            for result in results:
                lines.append(
                    result["title"] + ": " + result["body"][:300] + " (" + result["href"] + ")"
                )
            return "\n".join(lines)
        except Exception as error:
            if attempt == 1:
                return "Error: the search failed: " + type(error).__name__ + ": " + str(error)
            time.sleep(2)

    return "Error: the search failed."

### What the model sees
`@tool` turned our function into a tool object. These three lines are everything the model knows about it.

In [26]:
# TODO 2.2: print what the model sees of the tool: its name, its description and its inputs.
#   Hints: search_web.name      search_web.description      search_web.args

print("Name       :", search_web.name)
print("Description:", search_web.description)
print("Inputs     :", search_web.args)


Name       : search_web
Description: Search the web and return the top results with a short summary and a link for each.
Inputs     : {'query': {'title': 'Query', 'type': 'string'}}


### Using the tool by hand
We can call a tool ourselves with `.invoke(...)`. No model is involved.

In [27]:
# TODO 2.3: call the tool by hand with the query "Phewa Lake Pokhara" and print the result.
#   Hint: search_web.invoke({"query": "..."})

print(search_web.invoke({"query": "Phewa Lake Pokhara"}))

Phewa Lake - Wikipedia: Phewa Lake or Phewa Tal (Nepali: फेवा ताल, [ˈpʰewa tal]) is a freshwater lake in Nepal formerly called Baidam Tal located in the south of the Pokhara Valley that includes Pokhara city and parts of Sarangkot and Kaskikot. [1] (https://en.wikipedia.org/wiki/Phewa_Lake)
Phewa Lake Pokhara Travel Guide: What to See & Do: Aug 2, 2026 · Plan your visit to Phewa Lake with this guide to boating, temple visits, mountain views, seasonal timing, and how much time to allow. (https://walkthroughnepal.com/blog/phewa-lake-pokhara)
Phewa Lake: Experience Pokhara’s Breathtaking BeautyPhewa Tal - All You SHOULD Know Before Going (2026 Reviews)Phewa Lake, Pokhara: Boating, Tal Barahi & the Lakeside Shore ...Phewa Lake Pokhara – Boat Rides, Tal Barahi Temple ...Phewa Lake, फेवा ताल - Pokhara Travel Guide - Discover Nature ...: Mar 22, 2026 · Explore Phewa Lake in Pokhara, Nepal, with boating tips, Tal Barahi Temple, mountain reflections, best seasons, and local advice for a memorab

### When the search fails
A good tool does not crash. It **returns a short message starting with `Error:`** and the model can read it and react. Here we search for nothing (an empty query) to see it.

In [28]:
# TODO 2.4: call the tool with an EMPTY query ("") and print the result.
#   What does a good tool do when something goes wrong?

print(search_web.invoke({"query": "Phewa lake Pokhara"}))

Phewa Lake - Wikipedia: Phewa lake and water sports is one of the major tourist attractions of Pokhara city and the north shore of the lake has developed into a tourist district, commonly called Lakeside, with hotels, restaurants and bars catering to the tourists. [10] The water from Phewa lake's outlet is used to generate (https://en.wikipedia.org/wiki/Phewa_Lake)
Phewa Lake: Experience Pokhara's Breathtaking Beauty: Explore Phewa Lake in Pokhara, Nepal, with boating tips, Tal Barahi Temple, mountain reflections, best seasons, and local advice for a memorable visit. (https://nepalhikingadventure.com/phewa-lake/)
Phewa Lake Pokhara Travel Guide: What to See & Do: Plan your visit to Phewa Lake with this guide to boating, temple visits, mountain views, seasonal timing, and how much time to allow. (https://walkthroughnepal.com/blog/phewa-lake-pokhara)


---
# Part 3 - Workflow

Now we use the tool inside a **workflow**. **We** write the steps and their order. The model does the thinking inside two of them, and our code runs the search.

```
question  ->  [ LLM: write search words ]  ->  [ code: search_web ]  ->  [ LLM: write the answer ]
```

In [29]:
question = "What is the latest news about tourism in Nepal?"

# TODO 3.1 (Step 1, LLM): ask the model to write good search words for the question.
#   Tell it: 3 to 6 plain words, no quotes, no operators, no years, today is `today`, reply with ONLY the query.
#   Hint: search_words = llm.invoke("..." + question).content        then print("Step 1 - search words:", search_words)

# TODO 3.2 (Step 2, code): run the search tool with those words.
#   Hint: results = search_web.invoke({"query": search_words})        then print("Step 2 - search results:", results[:200])

# TODO 3.3 (Step 3, LLM): ask the model to answer the question using ONLY the results.
#   Tell it: if the results do not contain the answer, say so; answer in two or three short sentences; end with a link.
#   Hint: answer = llm.invoke("..." + question + "..." + results).content        then print("Step 3 - answer:", answer)

search_words = llm.invoke("Write the best web search query for this question, in 3 to 6 plain words. Use no quotes, no operators and no years. Today is " + today + ". Reply with ONLY the query. Question: " + question).content
print("Step 1 - search words:", search_words)

results = search_web.invoke({"query": search_words})
print("Step 2 - search results:", results[:200])

answer = llm.invoke("Answer the question using only these search results. If the results do not contain the answer, say so. Answer in two or three short sentences and end with a link.\n\nQuestion: " + question + "\n\nResults:\n" + results).content
print("Step 3 - answer:", answer)

Step 1 - search words: tourism news Nepal latest
Step 2 - search results: Error: the search failed: DDGSException: No results found.
Step 3 - answer: I’m sorry, but the provided search results do not contain any information about the latest news on tourism in Nepal. 

[No source available]


The order never changes: search words, search, answer. That makes it **predictable** and easy to test.

### It breaks when the question changes
This question needs **two** searches, one for each phone maker. Our workflow has only one search step, because we wrote it that way. Run the same three steps and read the answer.

In [30]:
question = "What are the newest flagship phones from Apple and from Samsung, and how much do they cost?"

# TODO 3.4: run the SAME three steps for this question, which needs TWO searches.
#   Copy your code from 3.1 to 3.3. Is the answer complete? Why not?

search_words = llm.invoke("Write the best web search query for this question, in 3 to 6 plain words. Use no quotes, no operators and no years. Today is " + today + ". Reply with ONLY the query. Question: " + question).content
print("Step 1 - search words:", search_words)

results = search_web.invoke({"query": search_words})
print("Step 2 - search results:", results[:200])

answer = llm.invoke("Answer the question using only these search results. If the results do not contain the answer, say so. Answer in two or three short sentences and end with a link.\n\nQuestion: " + question + "\n\nResults:\n" + results).content
print("Step 3 - answer:", answer)

Step 1 - search words: latest Apple Samsung flagship prices
Step 2 - search results: Samsung Galaxy S26 vs. Apple iPhone 17: The Definitive ...: Apr 7, 2026 · Apple and Samsung’s flagship phones are closer than ever. We pit the Galaxy S26 lineup up against the iPhone 17 family to comp
Step 3 - answer: The search results mention the latest flagship models as the Apple iPhone 17 and the Samsung Galaxy S26, but they do not provide the current pricing for either device. [Source]https://www.pcmag.com/comparisons/samsung-galaxy-s26-vs-apple-iphone-17-the-definitive-flagship-face-off


One search gave results about one topic (or a mix), so the answer is probably incomplete. To handle a question like this, **someone has to write another step**.

**Takeaway:** a workflow is reliable when you know the steps in advance. When you do not, you need something that can **decide** the steps. That is an agent, and we get there after seeing how a model uses a tool.

---
# Part 4 - Function calling

How does the model use a tool? It does **not** run it. It only **asks**. Our code runs the tool and hands the result back. This is called **function calling**, and it has three steps:

1. **Ask:** the model is told about the tool, and replies with a request.
2. **Run:** our code runs the tool with the inputs the model chose.
3. **Answer:** we give the result back, and the model writes the final answer.

### Step 1 - The model asks
`bind_tools` tells the model which tools exist. We also send a short **system message** with today's date, because a model does not know it. Look at what comes back: no answer, only a request.

In [ ]:
# TODO 4.1 (Step 1, the model asks): tell the model which tools exist, then ask a question.
#   - llm_with_tools = llm.bind_tools([search_web])
#   - system_message = SystemMessage(...) with: today's date, "do not add a year to your search words unless the user gave one",
#     and "when you answer, use only the tool result, in two or three short sentences, and end with the link of the page you used"
#   - question = "What is the latest news about tourism in Nepal?"
#   - reply = llm_with_tools.invoke([system_message, HumanMessage(question)])
#   - print reply.content (it should be empty) and reply.tool_calls

llm_with_tools = llm.bind_tools([search_web])

system_message = SystemMessage("Today's date is " + today + ". Do not add a year to your search words unless the user gave one. "
                               "When you answer, use only the tool result, in two or three short sentences, and end with the link of the page you used.")
question = "What is the latest news about tourism in Nepal?"
reply = llm_with_tools.invoke([system_message, HumanMessage(question)])

print("Text      :", reply.content)
print("Tool calls:", reply.tool_calls)

TypeError: 'str' object is not callable

The text is empty. `tool_calls` holds the request: the tool **name** and the **inputs** the model chose by reading the docstring. Nothing has run yet.

### Step 2 - We run the tool
We use the name and the inputs from the request.

In [ ]:
# TODO 4.2 (Step 2, we run the tool): take the first tool call from the reply and run the tool with its inputs.
#   Hints: call = reply.tool_calls[0]        call["name"]        call["args"]        call["id"]
#   search_result = search_web.invoke(call["args"])
#   Print the tool name and inputs the model asked for, then the first 300 characters of the result.

call = reply.tool_calls[0]
print("The model asked for:", call["name"], call["args"])

search_result = search_web.invoke(call["args"])
print("Our tool returned  :", search_result[:300])

The model asked for: search_web {'query': 'latest news tourism Nepal'}
Our tool returned  : Error: the search failed. Check the search words, or try again in a moment.


### Step 3 - We give the result back
We send the whole conversation: the same system message, the question, the model's request, and a `ToolMessage` with the result. The system message tells the model to answer **only from the tool result**. Without it, a model may add facts it made up. Now the model can write the answer.

In [ ]:
# TODO 4.3 (Step 3, we give the result back): build a list of messages, then ask the model for the final answer.
#   The list holds, in order:  system_message,  HumanMessage(question),  reply,  ToolMessage(content=search_result, tool_call_id=call["id"])
#   Hint: final = llm_with_tools.invoke(messages)        then print(final.content)

messages = [
    system_message,
    HumanMessage(question),
    reply,
    ToolMessage(content=search_result, tool_call_id=call["id"]),
]

final = llm_with_tools.invoke(messages)
print(final.content)

That is the whole idea: **ask, run, answer**. An agent does exactly these steps, in a loop, by itself.

---
# Part 5 - Agent

```
question  ->  [ LLM decides ]  <->  [ tools ]  ->  answer
```
An **agent** is a model plus tools plus a loop. `create_agent` builds all of it. We give it three things:

| Input | What it is |
|---|---|
| the model | the brain: `llm` |
| `tools` | what it is allowed to do |
| `system_prompt` | its job description |

First an agent with **no tools** (an empty list), to see the shape of the thing.

In [ ]:
# TODO 5.1: create an agent with NO tools: give it the model, an empty tools list and a short system_prompt.
#   Hint: agent = create_agent(llm, tools=[], system_prompt="...")
#   Then ask it "Search the web and tell me today's top news headline from Nepal." and print the answer.
#   Hint: result = agent.invoke({"messages": [("user", "...")]})        print(result["messages"][-1].content)

agent = create_agent(llm, tools=[], system_prompt="You are a helpful assistant. Answer in one or two short sentences.")

result = agent.invoke({"messages": [("user", "Search the web and tell me today's top news headline from Nepal.")]})

print(result["messages"][-1].content)

With no tools, the agent is only a model: it answers once, and it still cannot search. The result holds the whole conversation as a list of messages:

In [ ]:
# TODO 5.2: print every message in the result: its type and the first 100 characters of its content.
#   Hint: for message in result["messages"]:   then   message.type   and   message.content[:100]

for message in result["messages"]:
    print(message.type, "->", message.content[:100])

human -> Search the web and tell me today's top news headline from Nepal.
ai -> 


### Now we add the tool
We build the agent again, this time with `search_web`. We also write a proper **system prompt**: who the agent is, when to use the tool, and what to do when something goes wrong.

In [ ]:
# TODO 5.3: build the real agent and watch its steps.
#   1. system_prompt = a text (a job description) that says: you are a research assistant; today's date is `today`;
#      use the search_web tool and never guess; do not add a year to search words unless the user gave one;
#      your only tool is search_web and you cannot open web pages; write links as plain text, with no citation markers;
#      base the answer only on the tool and end with the link of the page you used; if the tool returns an Error, say so and
#      ask for different words; answer in two or three short sentences.
#   2. agent = create_agent(llm, tools=[search_web], system_prompt=system_prompt)
#   3. question = "What is the latest news about tourism in Nepal?"
#   4. print("> Entering agent..."), then loop over
#        agent.stream({"messages": [("user", question)]}, config={"recursion_limit": 10}, stream_mode="updates")
#      Each update is a dictionary:  for node, data in update.items():  message = data["messages"][-1]
#        - if node == "tools":   print("Observation:", message.content[:150])
#        - elif message.tool_calls:   for call in message.tool_calls:  print("Action:", call["name"])  and  print("Action Input:", call["args"])
#        - else:   print("Final Answer:", message.content)
#      Put the loop inside  try: ... except Exception as error: print("Stopped early:", str(error)[:200])
#   5. print("> Finished.")

system_prompt = ("You are a helpful research assistant. Today's date is " + today + ". "
                 "Use the search_web tool to look facts up and never guess. "
                 "Do not add a year to your search words unless the user gave one. "
                 "Your only tool is search_web. You cannot open web pages, so use only the search results. "
                 "Write the link as plain text, with no citation markers. "
                 "Base your answer only on what the tool returns, and end with the link of the page you used. "
                 "If the tool returns an Error, tell the user and ask for different words. "
                 "Answer in two or three short sentences.")

agent = create_agent(llm, tools=[search_web], system_prompt=system_prompt)

question = "What is the latest news about tourism in Nepal?"

print("> Entering agent...")
try:
    for update in agent.stream({"messages": [("user", question)]}, config={"recursion_limit": 10}, stream_mode="updates"):
        for node, data in update.items():
            message = data["messages"][-1]
            if node == "tools":
                print("Observation:", message.content[:150])
            elif message.tool_calls:
                for call in message.tool_calls:
                    print("Action:", call["name"])
                    print("Action Input:", call["args"])
            else:
                print("Final Answer:", message.content)
except Exception as error:
    print("Stopped early:", str(error)[:200])
print("> Finished.")

> Entering agent...
Action: search_web
Action Input: {'query': 'latest news tourism Nepal'}
Observation: Error: the search failed. Check the search words, or try again in a moment.
Action: search_web
Action Input: {'query': 'tourism Nepal news'}
Observation: The Tourism Times | Tourism Industry News in Nepal: 1 day ago · Get the latest on the tourism industry in Nepal, climbing, aviation, and hospitality n
Final Answer: The Tourism Times reports that Nepal’s tourism sector is seeing a surge in climbing, aviation and hospitality activity, with new initiatives aimed at boosting visitor numbers and improving infrastructure. For more details see the latest industry update on the Tourism Times website. https://thetourismtimes.com/
> Finished.


### Reading the steps
The agent printed its own loop, the same **ask, run, answer** we did by hand in Part 4:

| Line | Meaning |
|---|---|
| `Action` | Which **tool** the model chose |
| `Action Input` | The **inputs** it chose for that tool |
| `Observation` | What the tool **returned** |
| `Final Answer` | What the model wrote when it was done |

If the agent needs a second search, you will see a second `Action` and `Observation` before the final answer.

If you see `Stopped early`, the model made a mistake (for example, it asked for a tool it was not given) or Groq said "slow down". Run the cell again.

**With LangSmith tracing on**, the same run is also online. Open the project `day2-first-agent` at smith.langchain.com, click the newest run, and you will see the steps as a tree: `model`, `tools`, `model`. Click the `tools` step to see `search_web`, its inputs and what it returned.

### The question the workflow could not handle
Nobody wrote "search two times". The agent reads the question and decides how many searches it needs. `recursion_limit=10` in the loop is the safety net: it stops the agent after 10 steps, so it can never run forever.

In [ ]:
question = "What are the newest flagship phones from Apple and from Samsung, and how much do they cost?"

# TODO 5.4: the question the workflow could not handle. Run the SAME streaming loop as 5.3 with this question.
#   How many searches does the agent make? Is the answer complete now?

print("> Entering agent...")
try:
    for update in agent.stream({"messages": [("user", question)]}, config={"recursion_limit": 10}, stream_mode="updates"):
        for node, data in update.items():
            message = data["messages"][-1]
            if node == "tools":
                print("Observation:", message.content[:150])
            elif message.tool_calls:
                for call in message.tool_calls:
                    print("Action:", call["name"])
                    print("Action Input:", call["args"])
            else:
                print("Final Answer:", message.content)
except Exception as error:
    print("Stopped early:", str(error)[:200])
print("> Finished.")

> Entering agent...
Action: search_web
Action Input: {'query': 'newest flagship phone Apple cost'}
Observation: Error: the search failed. Check the search words, or try again in a moment.
Action: search_web
Action Input: {'query': 'Apple flagship phone price 2026'}
Observation: iPhone Price List 2026: Every Model and What to Pay: Apple's iPhone range has never had more price points, which is great for choice and confusing for
Action: search_web
Action Input: {'query': 'Samsung flagship phone price 2026'}
Observation: Samsung Flagship Mobile Phones (Oct 2026) | Beebom: Samsung Flagship Mobile Phones Explore the list of Samsung flagship mobile phones with standout fe
Action: search_web
Action Input: {'query': 'Samsung S26 Ultra price'}
Observation: Error: the search failed. Check the search words, or try again in a moment.
Action: search_web
Action Input: {'query': 'iPhone 18 Pro Max price'}
Observation: Error: the search failed. Check the search words, or try again in a moment.
Stopped 

### Give the agent a short-term memory
An agent can keep its messages between questions too. We give `create_agent` a **saver** (`InMemorySaver`), and every question carries a **thread id**: the name of the conversation. Same thread id, same memory.

In [ ]:
# TODO 5.5: give the agent a short-term memory.
#   - create the agent again, with one more argument:  checkpointer=InMemorySaver()
#   - config = {"configurable": {"thread_id": "class-1"}, "recursion_limit": 10}
#   - inside try / except: ask "What is the capital of Nepal?" with  agent.invoke({"messages": [("user", "...")]}, config)  and print the answer
#   - then ask "What is its population?" with the SAME config and print the answer. Did the agent understand "its"?
#   - print("Messages kept in this thread:", len(result["messages"]))

agent = create_agent(llm, tools=[search_web], system_prompt=system_prompt, checkpointer=InMemorySaver())

config = {"configurable": {"thread_id": "class-1"}, "recursion_limit": 10}

try:
    result = agent.invoke({"messages": [("user", "What is the capital of Nepal?")]}, config)
    print("Answer 1:", result["messages"][-1].content)

    result = agent.invoke({"messages": [("user", "What is its population?")]}, config)
    print("Answer 2:", result["messages"][-1].content)
    print("Messages kept in this thread:", len(result["messages"]))
except Exception as error:
    print("Stopped early:", str(error)[:200])

Answer 1: Kathmandu is the capital and largest city of Nepal. It serves as the country’s main governmental, financial, industrial, and cultural centre. https://en.wikipedia.org/wiki/Kathmandu
Answer 2: Kathmandu’s population is about 1.44 million people. https://worldstats.io/city/kathmandu-np
Messages kept in this thread: 8


The second question says "its", and the agent knew what it meant, because it still had the first question in its messages. Now the **same question in a new thread**:

In [ ]:
# TODO 5.6: ask "What is its population?" again, but in a NEW thread.
#   - other_thread = {"configurable": {"thread_id": "class-2"}, "recursion_limit": 10}
#   - agent.invoke(..., other_thread) inside try / except, then print the answer
#   What does the agent do without the history?

other_thread = {"configurable": {"thread_id": "class-2"}, "recursion_limit": 10}

try:
    result = agent.invoke({"messages": [("user", "What is its population?")]}, other_thread)
    print("New thread:", result["messages"][-1].content)
except Exception as error:
    print("Stopped early:", str(error)[:200])

New thread: I’m not sure which place or entity you’re asking about. Could you please tell me the name of the location or subject whose population you’d like to know?


A new thread has no history, so the agent has to guess or ask. Memory belongs to a **thread**. `InMemorySaver` keeps it only while this notebook is running: restart the notebook and it is gone.

---
# Part 6 - Your turn

Now **you** code. There are four tasks. Each has a **TODO**: a gap for you to fill. Fill the gap, run the cell, then run the **check** cell under it. The check tells you `PASS` or `NOT YET` and gives a hint.

| Task | You will |
|---|---|
| **1** | Write a tool called `multiply` |
| **2** | Give the agent both tools and ask a question that needs both |
| **3** | Read the agent's messages and list the tools it used, in order |
| **4** | Create your own tool |

### Task 1 - Write the `multiply` tool
A tool needs three things: `@tool` on top, a **docstring** the model reads, and a return value that is **text**. Fill in the gaps.

In [ ]:
@tool
def multiply(a: float, b: float) -> str:
    """TODO: write one sentence that tells the model what this tool does."""
    # TODO: work out a times b and return it as text, for example str(a * b)
    return "TODO"

In [ ]:
answer = multiply.invoke({"a": 17, "b": 19})
print("Your tool returned:", answer)

if "TODO" in multiply.description:
    print("NOT YET: write the docstring. The model reads it to decide when to use the tool.")
elif answer in ["323", "323.0"]:
    print("PASS: multiply works. 17 times 19 is 323.")
else:
    print("NOT YET: 17 times 19 should give 323. Return the product as text, for example str(a * b).")

Your tool returned: TODO
NOT YET: write the docstring. The model reads it to decide when to use the tool.


### Task 2 - Give the agent both tools
The agent below only has `search_web`. Add `multiply` to its list of tools, then run the cell. The answer should be about 26,500 (three times the height of Mount Everest in metres).

In [ ]:
lab_prompt = ("You are a helpful research assistant. Today's date is " + today + ". "
              "Use search_web to look facts up and multiply for exact maths, and never guess. "
              "You cannot open web pages. Write links as plain text. Answer in two or three short sentences.")

# TODO: add multiply to the list of tools
agent = create_agent(llm, tools=[search_web], system_prompt=lab_prompt)

question = "Search the web for the height of Mount Everest in metres, then multiply it by 3."

result = {"messages": []}
try:
    result = agent.invoke({"messages": [("user", question)]}, config={"recursion_limit": 12})
    print(result["messages"][-1].content)
except Exception as error:
    print("Stopped early:", str(error)[:200])

Mount Everest’s officially recognized height is 8,848.86 metres. Multiplying that by 3 gives 26,546.58 metres.


### Task 3 - Read the messages
The agent's result holds every message of the run, in order (this is the agent's **state**). A message of type `tool` is a tool result, and it has the tool's `name`. Fill in the gap to build a list of the tools the agent used, in order.

In [ ]:
tools_called = []
for message in result["messages"]:
    # TODO: if this message is a tool result (message.type == "tool"), add its name (message.name) to tools_called
    pass

print("Tools called:", tools_called)

Tools called: []


In [ ]:
if len(tools_called) == 0:
    print("NOT YET: tools_called is empty. Finish the loop above, and make sure Task 2 ran.")
elif "search_web" in tools_called and "multiply" in tools_called:
    print("PASS: the agent used", tools_called)
else:
    print("NOT YET: the agent used", tools_called, "but it should use both search_web and multiply. Is multiply in the tools list in Task 2? Run Task 2 again.")

NOT YET: tools_called is empty. Finish the loop above, and make sure Task 2 ran.


### Task 4 - Create your own tool
Write a tool of your own. Ideas: `celsius_to_fahrenheit(celsius: float)`, `word_count(text: str)`, `count_vowels(text: str)`.

Rules: `@tool` on top, simple inputs (`str` or `float`), a one-sentence docstring, and **return text**. Name your function `my_tool` so the check below can find it.

In [ ]:
# TODO: write your own tool here. Start with @tool, then def my_tool(...) -> str:

In [ ]:
try:
    print("Name       :", my_tool.name)
    print("Description:", my_tool.description)
    print("Inputs     :", my_tool.args)
    if "TODO" in my_tool.description or len(my_tool.description) < 10:
        print("NOT YET: write a clear one-sentence docstring.")
    else:
        print("PASS: you made a tool. Next, add it to an agent and ask a question that needs it.")
except NameError:
    print("NOT YET: write a tool and call the function my_tool.")

NOT YET: write a tool and call the function my_tool.


---
## What we learned
- A **basic LLM** answers from training only. It has no live data.
- A **workflow** runs steps that **we** wrote, in a fixed order. It is predictable, but it cannot handle a step nobody wrote.
- An **agent** is a model plus tools plus a loop, built with `create_agent`. **The model decides the steps.**
- A **tool** is a Python function with `@tool`. The model sees its name, docstring and inputs.
- **Function calling** is: the model **asks**, our code **runs**, the model **answers**.
- **Short-term memory** is the earlier messages, sent again. For an LLM we keep the list ourselves. For an agent we use a saver and a thread id.
- A good **system prompt** and **safe tools** (errors returned as text, like `Error: the search failed`) make an agent reliable.

## More ideas
1. **Change the system prompt.** Make the agent answer as a pirate. Does the tool still get used?
2. **Search settings.** Change `max_results=3` to 5 in the tool. Does the agent give a better answer? What does it cost in tokens?
3. **If a search fails.** The free search can say "slow down" when many people use it at once. Ask the same question twice in a row and read the `Error:` message if you get one.
4. **Use your own tool.** Add `my_tool` from Task 4 to an agent and ask a question that needs it.